# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ujjwalupreti/flyrank-internship-capstone/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [13]:
# Install required libraries
%pip -q install duckdb huggingface_hub pandas scikit-learn

import duckdb
import pandas as pd
from google.colab import userdata

# Authenticate and connect
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# Corrected paths: fact table is a partitioned folder, dim table is a single file
REL_FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
REL_DIM = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

# Build the feature vector using SQL to join facts and dimensions
df_features = con.sql(f"""
    SELECT
        f.content_hash_id,
        MAX(f.gsc_impressions) AS max_impressions_last_30d,
        SUM(f.gsc_clicks) AS clicks_last_30d,
        ANY_VALUE(d.content_type) AS content_type,

        -- Creating a mock label based on traffic drops for the trap
        CASE WHEN SUM(f.gsc_clicks) < 10 THEN 1 ELSE 0 END AS is_declining_label,

        -- Intentional leaky features preserved for the Section 3 test
        CASE WHEN SUM(f.gsc_clicks) < 10 THEN 'down' ELSE 'stable' END AS trend_direction
    FROM {REL_FACT} f
    LEFT JOIN {REL_DIM} d ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_impressions > 0
    GROUP BY f.content_hash_id
    LIMIT 10000
""").df()

# Fill any missing categorical values before encoding to avoid errors
df_features['content_type'] = df_features['content_type'].fillna('unknown')

# Handle Categorical Data: One-hot encode the 'content_type'
df_features = pd.get_dummies(df_features, columns=['content_type'], drop_first=True)

print(f"Feature vector successfully built. Shape: {df_features.shape}")
display(df_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector successfully built. Shape: (10000, 7)


,content_hash_id,max_impressions_last_30d,clicks_last_30d,is_declining_label,trend_direction,content_type_feedly article,content_type_keyword article
0,content_7a105f548d9c6916,555,7.0,1,down,False,True
1,content_a3ea9792f793ec72,26,0.0,1,down,False,True
2,content_36c36abc7650d7af,338,6.0,1,down,False,True
3,content_a7da352b73b02668,285,13.0,0,stable,False,True
4,content_1855a661b4d36130,27,1.0,1,down,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

* **`content_age_days`**: Age of the page in days. Missing values are coerced to 0. Available strictly before the prediction moment (static from publish date).
* **`impressions_last_30d` / `clicks_last_30d` / `ctr_last_30d**`: Traffic engagement metrics. Missing values are filled with 0 (assuming no traffic). These are trailing 30-day aggregates, fully logged and knowable before the decision moment.
* **`content_type`**: The categorical format of the page. Missing values drop out during one-hot encoding (`drop_first=True`). Available at the moment of publication.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
import numpy as np

df_features['leaky_feature'] = np.where(df_features['trend_direction'] == 'down', 1, 0)

clean_features = [col for col in df_features.columns if col not in ['content_hash_id', 'is_declining_label', 'trend_direction', 'leaky_feature', 'clicks_last_30d']]
leaky_features = clean_features + ['leaky_feature']

y = df_features['is_declining_label']

Xc_train, Xc_test, yc_train, yc_test = train_test_split(df_features[clean_features], y, test_size=0.2, random_state=42)
Xl_train, Xl_test, yl_train, yl_test = train_test_split(df_features[leaky_features], y, test_size=0.2, random_state=42)

clf_clean = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_clean.fit(Xc_train, yc_train)
clean_score = precision_score(yc_test, clf_clean.predict(Xc_test), zero_division=0)

clf_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_leaky.fit(Xl_train, yl_train)
leaky_score = precision_score(yl_test, clf_leaky.predict(Xl_test), zero_division=0)

print("--- Leakage Hunt Results ---")
print(f"Clean Model Precision: {clean_score:.2f}")
print(f"Leaky Model Precision: {leaky_score:.2f}")
print("\nVerdict: The leaky model's score jumps artificially because 'leaky_feature' is computed from the exact same logic as 'is_declining_label'. The model learns the rule, not the world[cite: 17].")

df_features = df_features.drop(columns=['trend_direction', 'leaky_feature'])
print("Leaky columns removed. Final frame secured.")

--- Leakage Hunt Results ---
Clean Model Precision: 0.91
Leaky Model Precision: 1.00

Verdict: The leaky model's score jumps artificially because 'leaky_feature' is computed from the exact same logic as 'is_declining_label'. The model learns the rule, not the world[cite: 17].
Leaky columns removed. Final frame secured.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* `trend_direction` and `trend_pct`: Excluded because they are label-derived columns. Our `is_declining_label` is computed directly from them. Including them allows the model to read the answer during training, causing massive data leakage.

* `client_id` and URLs: Excluded to ensure data privacy and to prevent the model from memorizing specific high-performing clients. The goal is to learn generalizable patterns about content, not to memorize client IDs.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.